# 02. Prompt + LLM
이 노트북은 `prompt_llm` 모드를 단독으로 실험합니다.

프로젝트 경로를 설정하고 STREAM 공통 모듈을 import합니다.

In [20]:
from pathlib import Path
import sys

WORKDIR = Path.cwd()
REPO_ROOT = WORKDIR.parent if WORKDIR.name == "notebooks" else WORKDIR
if REPO_ROOT.name == "Root_Stream":
    REPO_ROOT = REPO_ROOT.parent
PROJECT_ROOT = REPO_ROOT / "Root_Stream"

for path_item in (REPO_ROOT, PROJECT_ROOT):
    if str(path_item) not in sys.path:
        sys.path.append(str(path_item))

from Root_Stream.utils.config_loader import load_config
from Root_Stream.utils.path_utils import resolve_path
from Root_Stream.prompts.prompt_manager import PromptManager
from Root_Stream.orchestrator.stream_orchestrator import StreamOrchestrator


사용자 질문을 입력합니다.

In [21]:
user_question = "SELECT TOP (1000) * FROM [SVM3PRISM].[dbo].[EVENT_LOG_DATA];"
user_question


'SELECT TOP (1000) * FROM [SVM3PRISM].[dbo].[EVENT_LOG_DATA];'

프롬프트 템플릿 + LLM으로 SQL 문자열(query)을 생성합니다.

In [22]:
config_path = PROJECT_ROOT / "config" / "config.yaml"
config = load_config(config_path)
project_root = resolve_path(config.get("paths", {}).get("project_root", "."), PROJECT_ROOT)
prompt_file = resolve_path(config["paths"]["prompt_file"], project_root)
prompt_manager = PromptManager(prompt_file)

config["mode"] = "prompt_llm"
config.setdefault("prompts", {})["active_prompt"] = "query_generation_prompt"

# Fallback when user_question cell was not executed
if "user_question" not in globals() or not str(user_question).strip():
    user_question = "Show error counts by equipment in the last 30 days"

prompt_key = str(config.get("prompts", {}).get("active_prompt", "query_generation_prompt")).strip()
available_prompt_keys = prompt_manager.list_prompt_keys()
if prompt_key not in available_prompt_keys:
    print("[WARN] Invalid active_prompt. Fallback to query_generation_prompt:", prompt_key)
    prompt_key = "query_generation_prompt"

prompt_values = {
    "question": str(user_question),
    "schema_context": "",
    "retrieved_context": "",
    "business_rules": "",
    "additional_constraints": "",
    "api_result": "",
}
final_prompt = prompt_manager.render_prompt(prompt_key, prompt_values)
print("active_prompt:", prompt_key)
print("available_prompt_keys:", available_prompt_keys)
print("----- FINAL PROMPT -----")
print(final_prompt)

orchestrator = StreamOrchestrator(config=config, prompt_manager=prompt_manager, project_root=project_root)
result = orchestrator.run(str(user_question))
result.to_dict()


[2026-04-09 10:29:36] [INFO] [config_loader.py:load_config:38] 설정 로드 시작: c:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\config\config.yaml
[2026-04-09 10:29:36] [INFO] [config_loader.py:load_config:51] 설정 로드 완료
[2026-04-09 10:29:36] [INFO] [prompt_manager.py:_load_prompts:57] 프롬프트 파일 로드 시작: C:\Users\김민한\Desktop\개발\DB_to_LLM\Root_Stream\prompts\prompt_templates.yaml
[2026-04-09 10:29:36] [INFO] [prompt_manager.py:_load_prompts:70] 프롬프트 파일 로드 완료: prompt_count=3
[2026-04-09 10:29:36] [INFO] [stream_orchestrator.py:run_request:43] STREAM 실행 시작: mode=prompt_llm
[2026-04-09 10:29:36] [INFO] [mode_prompt_llm.py:run_prompt_llm_mode:65] mode 실행 시작: prompt_llm
[2026-04-09 10:29:36] [INFO] [ollama_client.py:generate:122] Ollama 호출 시작: model=qwen2.5:14b, endpoint=http://192.168.0.74:11434/api/chat


active_prompt: query_generation_prompt
available_prompt_keys: ['default_system_prompt', 'query_generation_prompt', 'rag_query_generation_prompt']
----- FINAL PROMPT -----
너는 Microsoft SQL Server(MSSQL) 전용 Text-to-SQL 질의 생성 전문가다.
사용자의 자연어 질문을 분석하여 실제 실행 가능한 MSSQL SELECT 쿼리를 생성하라.

[절대 원칙]
1. 반드시 MSSQL 문법만 사용한다.
2. 반드시 SELECT 쿼리만 생성한다.
3. INSERT, UPDATE, DELETE, MERGE, DROP, ALTER, TRUNCATE, CREATE, EXEC 등 변경 구문은 절대 생성하지 않는다.
4. schema_context에 없는 테이블명, 컬럼명은 절대 사용하지 않는다.
5. 과도한 추측을 하지 않는다.
6. 출력은 SQL 본문만 반환한다.
7. 설명, 코드블록, 주석은 절대 포함하지 않는다.

[MSSQL 규칙]
- LIMIT 대신 TOP을 사용한다.
- 날짜 계산은 DATEADD, DATEDIFF를 사용한다.
- 현재 시각 기준은 GETDATE()를 사용한다.
- 날짜 변환은 CAST 또는 CONVERT를 사용한다.
- 집계 쿼리에서는 GROUP BY를 정확히 작성한다.
- TOP 사용 시 필요한 경우 ORDER BY를 함께 사용한다.

[사용자 질문]
SELECT TOP (1000) * FROM [SVM3PRISM].[dbo].[EVENT_LOG_DATA];

[스키마 정보]


[업무 규칙]


[추가 제약]


[생성 지침]
- 질문이 목록 조회인지, 단건 조회인지, 집계 조회인지 먼저 판단하라.
- 필요한 테이블과 컬럼만 선택하라.
- JOIN은 명확한 관계가 있을 때만 사용하라.
- 기간 조건이 있으면 MSSQL 기준으로 안전하게 해석하라.
- 불필요한 SELECT *는 사용하지 마

[2026-04-09 10:29:37] [INFO] [ollama_client.py:generate:170] Ollama 호출 완료: output_length=59
[2026-04-09 10:29:37] [INFO] [mode_prompt_llm.py:run_prompt_llm_mode:99] mode 실행 완료: prompt_llm
[2026-04-09 10:29:37] [INFO] [stream_orchestrator.py:run_request:76] STREAM 실행 완료: mode=prompt_llm


{'mode': 'prompt_llm',
 'question': 'SELECT TOP (1000) * FROM [SVM3PRISM].[dbo].[EVENT_LOG_DATA];',
 'query': 'SELECT TOP (1000) * FROM [SVM3PRISM].[dbo].[EVENT_LOG_DATA]',
 'llm_provider': 'ollama',
 'prompt_key': 'query_generation_prompt',
 'retrieved_contexts': [],
 'raw_response': None,
 'metadata': {'mode': 'prompt_llm', 'llm_provider': 'ollama'}}

## MSSQL 실행 단계
위에서 생성된 `result.query`를 공통 SQL 실행 서비스로 안전하게 실행합니다.

In [23]:
from IPython.display import display

from Root_Stream.services.sql.sql_executor_service import SqlExecutorService
from Root_Stream.services.sql.sql_execution_integration import (
    build_execution_payload,
    run_generated_sql_with_executor,
)

# MSSQL connection (IP / PORT / DB / ID / PW)
db_host = "192.168.0.11"
db_port = 1433
db_name = "SVM3PRISM"
db_user = "prismadm"
db_password = "prism@2025"

database_config = config.setdefault("database", {})
database_config.update(
    {
        "type": "mssql",
        "host": db_host,
        "port": db_port,
        "database": db_name,
        "username": db_user,
        "password": db_password,
        "driver": database_config.get("driver", "ODBC Driver 17 for SQL Server"),
        "encrypt": bool(database_config.get("encrypt", False)),
        "trust_server_certificate": bool(database_config.get("trust_server_certificate", True)),
        "timeout": int(database_config.get("timeout", 30)),
    }
)

sql_config = config.setdefault("sql", {})
sql_config.setdefault("allow_only_select", True)
sql_config.setdefault("max_rows", 1000)

if "result" not in locals():
    raise ValueError("먼저 SQL 생성 셀을 실행해 result를 만든 뒤 이 셀을 실행하세요.")

executor = SqlExecutorService.from_config(config)
try:
    generated_sql = result.query
    df = run_generated_sql_with_executor(generated_sql=generated_sql, executor=executor)
    execution_payload = build_execution_payload(df)
finally:
    executor.close()

print(f"조회 row_count: {execution_payload['row_count']}")
print("조회 결과 표(최대 20행 미리보기)")
display(df.head(20))
execution_payload


[2026-04-09 10:29:41] [INFO] [sql_executor_service.py:from_config:59] SQL 실행 서비스 생성 완료: allow_only_select=True, max_rows=1000
[2026-04-09 10:29:41] [INFO] [sql_execution_integration.py:run_generated_sql_with_executor:44] 생성 SQL 실행 연결 시작
[2026-04-09 10:29:41] [INFO] [sql_executor_service.py:run_query:81] SQL 실행 요청 수신
[2026-04-09 10:29:41] [INFO] [sql_guard.py:validate_query_sql:59] SQL 검증 시작
[2026-04-09 10:29:41] [INFO] [sql_guard.py:validate_query_sql:73] SQL 검증 완료
[2026-04-09 10:29:41] [INFO] [sql_executor_service.py:run_query:83] MSSQL 조회 실행 시작
[2026-04-09 10:29:41] [INFO] [mssql_client.py:execute_query:110] MSSQL 조회 시작: host=192.168.0.11, database=SVM3PRISM
[2026-04-09 10:29:41] [INFO] [mssql_client.py:execute_query:119] MSSQL 조회 완료: row_count=1000
[2026-04-09 10:29:41] [INFO] [sql_executor_service.py:run_query:86] SQL 실행 완료: row_count=1000
[2026-04-09 10:29:41] [INFO] [sql_execution_integration.py:run_generated_sql_with_executor:46] 생성 SQL 실행 연결 완료: row_count=1000
[2026-04-09 10:29

조회 row_count: 1000
조회 결과 표(최대 20행 미리보기)


,EQPID,MODULEID,EVENTTIME,UNIT,STEP,GLASSID,GLASSID2,LOTID,HOSTPPID,EQPPPID,EVENTID,COL1,COL2,COL3,COL4,COL5,FILENAME,CREATETIME
0,S1JOA01D,S1JOA01D_NC01,2026-03-06 00:00:09.725,NC01,1005,S1FM1C5504H4S,,S1FM1C550AO0,PE_8766_FM1_PXL,PE_8766_FM1_PXL,CEID_201,MASK,EF3-R08-BML-C2,MATERIAL_STOCK_IN,NC01,NC01,T-S1JOA01D_NC01-20260306001603.csv,2026-03-05 15:16:08.235653
1,S1JOA01D,S1JOA01D_NC01,2026-03-06 00:00:32.550,NC01,1005,S1FM1C5504H4S,,S1FM1C550AO0,PE_8766_FM1_PXL,PE_8766_FM1_PXL,CEID_17,COMPONENT_OUT,NC01,NC01,,,T-S1JOA01D_NC01-20260306001603.csv,2026-03-05 15:16:08.235653
2,S1JOA01D,S1JOA01D_NC01,2026-03-06 00:00:37.861,NC01,1005,S1FM1C5504H4T,,S1FM1C550AO0,PE_8766_FM1_PXL,PE_8766_FM1_PXL,CEID_16,COMPONENT_IN,NC01,NC01,,,T-S1JOA01D_NC01-20260306001603.csv,2026-03-05 15:16:08.235653
3,S1JOA01D,S1JOA01D_NC01,2026-03-06 00:01:28.616,NC01,1005,S1FM1C5504H4T,,S1FM1C550AO0,PE_8766_FM1_PXL,PE_8766_FM1_PXL,CEID_17,COMPONENT_OUT,NC01,NC01,,,T-S1JOA01D_NC01-20260306001603.csv,2026-03-05 15:16:08.235653
4,S1JOA01D,S1JOA01D_NC01,2026-03-06 00:01:34.352,NC01,1005,S1FM1C5504H4U,,S1FM1C550AO0,PE_8766_FM1_PXL,PE_8766_FM1_PXL,CEID_16,COMPONENT_IN,NC01,NC01,,,T-S1JOA01D_NC01-20260306001603.csv,2026-03-05 15:16:08.235653
5,S1JOA01D,S1JOA01D_NC01,2026-03-06 00:02:25.425,NC01,1005,S1FM1C5504H4U,,S1FM1C550AO0,PE_8766_FM1_PXL,PE_8766_FM1_PXL,CEID_17,COMPONENT_OUT,NC01,NC01,,,T-S1JOA01D_NC01-20260306001603.csv,2026-03-05 15:16:08.235653
6,S1JOA01D,S1JOA01D_NC01,2026-03-06 00:02:31.112,NC01,1005,S1FM1C5504H4V,,S1FM1C550AO0,PE_8766_FM1_PXL,PE_8766_FM1_PXL,CEID_16,COMPONENT_IN,NC01,NC01,,,T-S1JOA01D_NC01-20260306001603.csv,2026-03-05 15:16:08.235653
7,S1JOA01D,S1JOA01D_NC01,2026-03-06 00:03:21.595,NC01,1005,S1FM1C5504H4V,,S1FM1C550AO0,PE_8766_FM1_PXL,PE_8766_FM1_PXL,CEID_17,COMPONENT_OUT,NC01,NC01,,,T-S1JOA01D_NC01-20260306001603.csv,2026-03-05 15:16:08.235653
8,S1JOA01D,S1JOA01D_NC01,2026-03-06 00:03:26.936,NC01,1005,S1FM1C5504H4W,,S1FM1C550AO0,PE_8766_FM1_PXL,PE_8766_FM1_PXL,CEID_16,COMPONENT_IN,NC01,NC01,,,T-S1JOA01D_NC01-20260306001603.csv,2026-03-05 15:16:08.235653
9,S1JOA01D,S1JOA01D_NC01,2026-03-06 00:04:17.314,NC01,1005,S1FM1C5504H4W,,S1FM1C550AO0,PE_8766_FM1_PXL,PE_8766_FM1_PXL,CEID_17,COMPONENT_OUT,NC01,NC01,,,T-S1JOA01D_NC01-20260306001603.csv,2026-03-05 15:16:08.235653


{'columns': ['EQPID',
  'MODULEID',
  'EVENTTIME',
  'UNIT',
  'STEP',
  'GLASSID',
  'GLASSID2',
  'LOTID',
  'HOSTPPID',
  'EQPPPID',
  'EVENTID',
  'COL1',
  'COL2',
  'COL3',
  'COL4',
  'COL5',
  'FILENAME',
  'CREATETIME'],
 'row_count': 1000,
 'rows': [{'EQPID': 'S1JOA01D',
   'MODULEID': 'S1JOA01D_NC01',
   'EVENTTIME': Timestamp('2026-03-06 00:00:09.725000'),
   'UNIT': 'NC01',
   'STEP': '1005',
   'GLASSID': 'S1FM1C5504H4S',
   'GLASSID2': '',
   'LOTID': 'S1FM1C550AO0',
   'HOSTPPID': 'PE_8766_FM1_PXL',
   'EQPPPID': 'PE_8766_FM1_PXL',
   'EVENTID': 'CEID_201',
   'COL1': 'MASK',
   'COL2': 'EF3-R08-BML-C2',
   'COL3': 'MATERIAL_STOCK_IN',
   'COL4': 'NC01',
   'COL5': 'NC01',
   'FILENAME': 'T-S1JOA01D_NC01-20260306001603.csv',
   'CREATETIME': Timestamp('2026-03-05 15:16:08.235653')},
  {'EQPID': 'S1JOA01D',
   'MODULEID': 'S1JOA01D_NC01',
   'EVENTTIME': Timestamp('2026-03-06 00:00:32.550000'),
   'UNIT': 'NC01',
   'STEP': '1005',
   'GLASSID': 'S1FM1C5504H4S',
   'GLAS